# Phase 1 — Data Audit & Extraction

## Life Claims Intelligence

### Objective

Audit the source data and extract the required Life Insurance Claims
metrics for FY2020-21 to FY2024-25.

This phase focuses on understanding the source structure, validating
the available data, and creating a consistent insurer-year level
analytical dataset.

### Workflow

1. Inspect the source data
2. Identify the required Life Claims tables and metrics
3. Extract insurer-level records
4. Validate financial-year coverage
5. Validate insurer-year uniqueness
6. Validate metric structure across years
7. Perform final data-quality checks
8. Export the consolidated raw dataset

### Data Integrity Principle

The original source data is treated as immutable.

No cleaning, imputation, feature engineering, or analytical transformation
is performed in this phase. The extracted dataset is saved separately
for downstream processing.

---

## 1. Import Libraries

The first step is to import the Python libraries required for data loading, inspection, and analysis.

In [1]:
import pandas as pd
import numpy as np

---

## 2. Load Source Workbook

The extracted IRDAI source tables are stored in the raw data layer.

The raw source workbook will be treated as immutable and will only be read for auditing and downstream processing.

In [2]:
from pathlib import Path

source_path = Path("../data/raw/Insurance_Claims_Intelligence_Source_Tables.xlsx")

source_path.exists()

True

In [3]:
xls = pd.ExcelFile(source_path)

xls.sheet_names

['SOURCE_INDEX',
 '2021_22_Life_Claims',
 '2022_23_Life_Claims',
 '2023_24_Life_Claims',
 '2024_25_Life_Claims',
 '2020_21_Life_Premium',
 '2021_22_Life_Premium',
 '2022_23_Life_Premium',
 '2023_24_Life_Premium',
 '2024_25_Life_Premium',
 '2020_21_Life_Grievances',
 '2021_22_Life_Grievances',
 '2022_23_Life_Grievances',
 '2023_24_Life_Grievances',
 '2024_25_Life_Grievances',
 '2020_21_Life_PolicyVolume',
 '2021_22_Life_PolicyVolume',
 '2022_23_Life_PolicyVolume',
 '2023_24_Life_PolicyVolume',
 '2024_25_Life_PolicyVolume',
 '2020_21_Life_Duration',
 '2021_22_Life_Duration',
 '2022_23_Life_Duration',
 '2023_24_Life_Duration',
 '2024_25_Life_Duration',
 '2020_21_Health_Claims',
 '2021_22_Health_Claims',
 '2022_23_Health_Claims',
 '2023_24_Health_Claims',
 '2024_25_Health_Claims',
 '2020_21_Health_Premium',
 '2021_22_Health_Premium',
 '2022_23_Health_Premium',
 '2023_24_Health_Premium',
 '2024_25_Health_Premium',
 '2020_21_Health_ICR',
 '2021_22_Health_ICR',
 '2022_23_Health_ICR',
 '2023_2

---

## 3. Source Index Audit

The SOURCE_INDEX sheet is used to verify the extracted IRDAI tables, their financial years, analytical tracks, original table references, and availability before beginning data preparation.

In [4]:
source_index = pd.read_excel(
    source_path,
    sheet_name="SOURCE_INDEX"
)

source_index.head()

,FY,Track,Data Category,Original Table Number,Original Table Title,Source Part,Sheet/Table extracted,Available?,Notes
0,2020-21,Life,Individual Death Claims - Insurer-wise,NaN,Individual Death Claims of Life Insurers - Ins...,NaN,NOT AVAILABLE,NO,NOT AVAILABLE: Table 13 in FY2020-21 is an AGG...
1,2021-22,Life,Individual Death Claims - Insurer-wise,16.0,Individual Death Claims of Life Insurers - Ins...,life,2021_22_Life_Claims,YES,NaN
2,2022-23,Life,Individual Death Claims - Insurer-wise,15.0,Individual Death Claims of Life Insurers - Ins...,life,2022_23_Life_Claims,YES,NaN
3,2023-24,Life,Individual Death Claims - Insurer-wise,15.0,Individual Death Claims of Life Insurers - Ins...,life,2023_24_Life_Claims,YES,NaN
4,2024-25,Life,Individual Death Claims - Insurer-wise,15.0,Individual Death Claims of Life Insurers - Ins...,life,2024_25_Life_Claims,YES,NaN


In [5]:
source_index.shape

(65, 9)

In [6]:
source_index.columns.tolist()

['FY',
 'Track',
 'Data Category',
 'Original Table Number',
 'Original Table Title',
 'Source Part',
 'Sheet/Table extracted',
 'Available?',
 'Notes']

----

## 4. Life Claims Source Table Inspection

Before cleaning or transforming the data, the insurer-wise Life claims source tables are inspected to understand their original structure, headers, historical periods, insurer rows, units, and any rolling-year data.

In [7]:
life_2021_22 = pd.read_excel(
    source_path,
    sheet_name="2021_22_Life_Claims",
    header=None
)

life_2021_22.shape

(73, 42)

In [8]:
life_2021_22.head(25)

,0,1,2,3,4,5,6,7,8,9,...,32,33,34,35,36,37,38,39,40,41
0,TABLE 16: INDIVIDUAL DEATH CLAIMS OF LIFE INSU...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S.No.,Insurer,2020-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Claims pending at start of the period,NaN,Claims intimated / booked,NaN,Total Claims,NaN,Claims Paid,NaN,...,Claims rejected,NaN,Claims Unclaimed,NaN,Claims pending at end of the period,NaN,Break up of claims pending - duration wise \n(...,NaN,NaN,NaN
3,NaN,NaN,No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),...,No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),< 3 months,3 - < 6 months,6 - <1 yr,> 1 yr
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,Public Sector,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,LIC,5875,349.69,941101,18755.65,946976,19105.34,933889,18295.58,...,3619,8.5,2625,461.66,2282,360.15,1211,1071,-,-
7,NaN,NaN,NaN,NaN,NaN,NaN,1,1.0,0.98618,0.957616,...,0.002647,0.000287,0.00192,0.015604,0.001669,0.012173,0.530675,0.469325,-,-
8,NaN,Private Sector,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2,Aditya Birla Sun Life,19,3.802883,6455,468.846357,6474,472.64924,6347,440.264288,...,-,-,-,-,7,0.907645,7,-,-,-


In [9]:
life_2021_22.iloc[:15, :20]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,TABLE 16: INDIVIDUAL DEATH CLAIMS OF LIFE INSU...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S.No.,Insurer,2020-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Claims pending at start of the period,NaN,Claims intimated / booked,NaN,Total Claims,NaN,Claims Paid,NaN,Claims Repudiated,NaN,Claims Rejected,NaN,Claims Unclaimed,NaN,Claims pending at end of the period,NaN,Break up of claims pending -- duration wise (P...,NaN
3,NaN,NaN,No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),< 3 months,3 - < 6 months
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,Public Sector,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,LIC,5875,349.69,941101,18755.65,946976,19105.34,933889,18295.58,6531,276.93,2934,3.92,1897,236.49,1725,292.42,792,933
7,NaN,NaN,NaN,NaN,NaN,NaN,1,1.0,0.98618,0.957616,0.006897,0.014495,0.003098,0.000205,0.002003,0.012378,0.001822,0.015306,0.45913,0.54087
8,NaN,Private Sector,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2,Aditya Birla Sun Life,19,3.802883,6455,468.846357,6474,472.64924,6347,440.264288,116,28.747045,0,0,0,0,11,3.637908,10,1


In [12]:
life_2021_22.iloc[:5, :].T

,0,1,2,3,4
0,TABLE 16: INDIVIDUAL DEATH CLAIMS OF LIFE INSU...,S.No.,NaN,NaN,NaN
1,NaN,Insurer,NaN,NaN,NaN
2,NaN,2020-21,Claims pending at start of the period,No. of Policies,NaN
3,NaN,NaN,NaN,Benefit Amount (₹crore),NaN
4,NaN,NaN,Claims intimated / booked,No. of Policies,NaN
5,NaN,NaN,NaN,Benefit Amount (₹crore),NaN
6,NaN,NaN,Total Claims,No. of Policies,NaN
7,NaN,NaN,NaN,Benefit Amount (₹crore),NaN
8,NaN,NaN,Claims Paid,No. of Policies,NaN
9,NaN,NaN,NaN,Benefit Amount (₹crore),NaN


In [13]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

life_2021_22.iloc[:6, :]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41
0,TABLE 16: INDIVIDUAL DEATH CLAIMS OF LIFE INSU...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S.No.,Insurer,2020-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Claims pending at start of the period,NaN,Claims intimated / booked,NaN,Total Claims,NaN,Claims Paid,NaN,Claims Repudiated,NaN,Claims Rejected,NaN,Claims Unclaimed,NaN,Claims pending at end of the period,NaN,Break up of claims pending -- duration wise (P...,NaN,NaN,NaN,Claims pending at start of the period,NaN,Claims intimated / booked,NaN,Total Claims,NaN,Claims paid,NaN,Claims Repudiated,NaN,Claims rejected,NaN,Claims Unclaimed,NaN,Claims pending at end of the period,NaN,Break up of claims pending - duration wise \n(...,NaN,NaN,NaN
3,NaN,NaN,No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),< 3 months,3 - < 6 months,6 - <1 yr,> 1 yr,No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),No. of Policies,Benefit Amount (₹crore),< 3 months,3 - < 6 months,6 - <1 yr,> 1 yr
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,Public Sector,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


----

## 5. Column Structure Mapping

The source table contains multiple header rows and paired columns for claim counts and benefit amounts.

This section maps each source column to its original meaning before any cleaning or transformation is performed.

In [14]:
for col in life_2021_22.columns:
    print(
        f"Column {col}:",
        list(life_2021_22.iloc[:6, col])
    )

Column 0: ['TABLE 16: INDIVIDUAL DEATH CLAIMS OF LIFE INSURERS - INSURER-WISE', 'S.No.', nan, nan, nan, nan]
Column 1: [nan, 'Insurer', nan, nan, nan, 'Public Sector']
Column 2: [nan, '2020-21', 'Claims pending at start of the period', 'No. of Policies', nan, nan]
Column 3: [nan, nan, nan, 'Benefit Amount (₹crore)', nan, nan]
Column 4: [nan, nan, 'Claims intimated / booked', 'No. of Policies', nan, nan]
Column 5: [nan, nan, nan, 'Benefit Amount (₹crore)', nan, nan]
Column 6: [nan, nan, 'Total Claims', 'No. of Policies', nan, nan]
Column 7: [nan, nan, nan, 'Benefit Amount (₹crore)', nan, nan]
Column 8: [nan, nan, 'Claims Paid', 'No. of Policies', nan, nan]
Column 9: [nan, nan, nan, 'Benefit Amount (₹crore)', nan, nan]
Column 10: [nan, nan, 'Claims Repudiated ', 'No. of Policies', nan, nan]
Column 11: [nan, nan, nan, 'Benefit Amount (₹crore)', nan, nan]
Column 12: [nan, nan, 'Claims Rejected', 'No. of Policies', nan, nan]
Column 13: [nan, nan, nan, 'Benefit Amount (₹crore)', nan, nan]
Co

In [15]:
for i in range(22, 42):
    print(
        i,
        "|",
        life_2021_22.iloc[1, i],
        "|",
        life_2021_22.iloc[2, i],
        "|",
        life_2021_22.iloc[3, i]
    )

22 | 2021-22 | Claims pending at start of the period | No. of Policies
23 | nan | nan | Benefit Amount (₹crore)
24 | nan | Claims intimated / booked | No. of Policies
25 | nan | nan | Benefit Amount (₹crore)
26 | nan | Total Claims | No. of Policies
27 | nan | nan | Benefit Amount (₹crore)
28 | nan | Claims paid | No. of Policies
29 | nan | nan | Benefit Amount (₹crore)
30 | nan | Claims Repudiated  | No. of Policies
31 | nan | nan | Benefit Amount (₹crore)
32 | nan | Claims rejected | No. of Policies
33 | nan | nan | Benefit Amount (₹crore)
34 | nan | Claims Unclaimed | No. of Policies
35 | nan | nan | Benefit Amount (₹crore)
36 | nan | Claims pending at end of the period | No. of Policies
37 | nan | nan | Benefit Amount (₹crore)
38 | nan | Break up of claims pending - duration wise 
(No. Policies) | < 3 months
39 | nan | nan | 3 - < 6 months
40 | nan | nan | 6 - <1 yr
41 | nan | nan | > 1 yr


In [16]:
life_2021_22.iloc[:, [0, 1]].head(20)

,0,1
0,TABLE 16: INDIVIDUAL DEATH CLAIMS OF LIFE INSU...,NaN
1,S.No.,Insurer
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
5,NaN,Public Sector
6,1,LIC
7,NaN,NaN
8,NaN,Private Sector
9,2,Aditya Birla Sun Life


In [17]:
life_2021_22.iloc[:, [0, 1]].tail(20)

,0,1
53,24,Tata AIA
54,NaN,NaN
55,NaN,Private Sector Total
56,NaN,NaN
57,NaN,Grand Total
58,NaN,NaN
59,Note: First row across each insurer shows the ...,NaN
60,NaN,NaN
61,NaN,NaN
62,NaN,NaN


In [18]:
life_2021_22.iloc[:, 1].dropna().tolist()

['Insurer',
 'Public Sector',
 'LIC ',
 'Private Sector',
 'Aditya Birla Sun Life',
 'Aegon',
 'Ageas Federal',
 'Aviva',
 'Bajaj Allianz',
 'Bharti Axa ',
 'Canara HSBC OBC',
 'Edelweiss Tokio',
 'Exide Life',
 'Future Generali',
 'HDFC Life',
 'ICICI Prudential',
 'India First',
 'Kotak Mahindra ',
 'Max Life',
 'PNB Met Life',
 'Pramerica Life',
 'Reliance Nippon',
 'Sahara',
 'SBI Life ',
 'Shriram',
 'Star Union',
 'Tata AIA',
 'Private Sector Total',
 'Grand Total']

----

## 6. Row Structure Audit

The IRDAI source table contains insurer-level records followed by percentage rows, along with sector totals, grand totals, and notes.

These rows must be identified correctly before extracting the insurer-level observations. No rows are removed at this stage.

In [19]:
for i in range(5, 60):
    print(
        i,
        "|",
        life_2021_22.iloc[i, 1],
        "|",
        life_2021_22.iloc[i, 2],
        "|",
        life_2021_22.iloc[i, 3]
    )

5 | Public Sector | nan | nan
6 | LIC  | 5875 | 349.6899999999998
7 | nan | nan | nan
8 | Private Sector | nan | nan
9 | Aditya Birla Sun Life | 19 | 3.802882696
10 | nan | nan | nan
11 | Aegon | 0 | 0
12 | nan | nan | nan
13 | Ageas Federal | 5 | 1.255
14 | nan | nan | nan
15 | Aviva | 5 | 0.7783592
16 | nan | nan | nan
17 | Bajaj Allianz | 2 | 0.650000034999987
18 | nan | nan | nan
19 | Bharti Axa  | 2 | 1.1533918
20 | nan | nan | nan
21 | Canara HSBC OBC | 2 | 2
22 | nan | nan | nan
23 | Edelweiss Tokio | 0 | 0
24 | nan | nan | nan
25 | Exide Life | 38 | 9.048578000000006
26 | nan | nan | nan
27 | Future Generali | 3 | 0.31135
28 | nan | nan | nan
29 | HDFC Life | 35 | 17.578717205
30 | nan | nan | nan
31 | ICICI Prudential | 95 | 43.70162686145701
32 | nan | nan | nan
33 | India First | 9 | 2.531358599
34 | nan | nan | nan
35 | Kotak Mahindra  | 9 | 12.142200569
36 | nan | nan | nan
37 | Max Life | 1 | 0.01500105
38 | nan | nan | nan
39 | PNB Met Life | 0 | 0
40 | nan | nan | nan
4

### final verification

In [20]:
life_2021_22.iloc[6:8, :22].T

,6,7
0,1,NaN
1,LIC,NaN
2,5875,NaN
3,349.69,NaN
4,941101,NaN
5,18755.65,NaN
6,946976,1
7,19105.34,1.0
8,933889,0.98618
9,18295.58,0.957616


In [21]:
life_2021_22.iloc[9:11, 22:42].T

,9,10
22,11,NaN
23,3.637908,NaN
24,9997,NaN
25,876.685045,NaN
26,10008,1
27,880.322952,1
28,9815,0.980715
29,846.434314,0.961504
30,186,0.018585
31,32.980993,0.037465


----

## 7. Multi-Year Structure Validation

The Life Claims tables are published with rolling historical periods. This section compares the five source sheets to verify the availability and structural consistency of the required insurer-level claims fields across FY2020–21 to FY2024–25.

In [22]:
life_sheets = {
    "2021-22": "2021_22_Life_Claims",
    "2022-23": "2022_23_Life_Claims",
    "2023-24": "2023_24_Life_Claims",
    "2024-25": "2024_25_Life_Claims"
}

life_raw = {}

for year, sheet in life_sheets.items():
    life_raw[year] = pd.read_excel(
        source_path,
        sheet_name=sheet,
        header=None
    )

for year, df in life_raw.items():
    print(year, "→", df.shape)

2021-22 → (73, 42)
2022-23 → (72, 62)
2023-24 → (76, 82)
2024-25 → (79, 82)


----

## 8. Cross-Year Header Structure Audit

The number of columns differs across the Life Claims source sheets. This section inspects the header hierarchy of each year to identify the financial-year blocks and verify the location of the required claims metrics before extraction.

In [23]:
for year, df in life_raw.items():
    print("=" * 80)
    print(f"{year} | Shape: {df.shape}")
    print("=" * 80)

    for col in range(df.shape[1]):
        values = df.iloc[:4, col].tolist()

        if any(pd.notna(v) for v in values):
            print(col, "|", values)

    print("\n")

2021-22 | Shape: (73, 42)
0 | ['TABLE 16: INDIVIDUAL DEATH CLAIMS OF LIFE INSURERS - INSURER-WISE', 'S.No.', nan, nan]
1 | [nan, 'Insurer', nan, nan]
2 | [nan, '2020-21', 'Claims pending at start of the period', 'No. of Policies']
3 | [nan, nan, nan, 'Benefit Amount (₹crore)']
4 | [nan, nan, 'Claims intimated / booked', 'No. of Policies']
5 | [nan, nan, nan, 'Benefit Amount (₹crore)']
6 | [nan, nan, 'Total Claims', 'No. of Policies']
7 | [nan, nan, nan, 'Benefit Amount (₹crore)']
8 | [nan, nan, 'Claims Paid', 'No. of Policies']
9 | [nan, nan, nan, 'Benefit Amount (₹crore)']
10 | [nan, nan, 'Claims Repudiated ', 'No. of Policies']
11 | [nan, nan, nan, 'Benefit Amount (₹crore)']
12 | [nan, nan, 'Claims Rejected', 'No. of Policies']
13 | [nan, nan, nan, 'Benefit Amount (₹crore)']
14 | [nan, nan, 'Claims Unclaimed', 'No. of Policies']
15 | [nan, nan, nan, 'Benefit Amount (₹crore)']
16 | [nan, nan, 'Claims pending at end of the period', 'No. of Policies']
17 | [nan, nan, nan, 'Benefit Amoun

In [24]:
import re

for year, df in life_raw.items():
    print("=" * 70)
    print(f"{year} | Shape: {df.shape}")
    
    for col in range(df.shape[1]):
        for row in range(min(4, df.shape[0])):
            value = df.iloc[row, col]
            
            if pd.notna(value):
                text = str(value).strip()
                
                if re.fullmatch(r"\d{4}-\d{2}", text):
                    print(f"Column {col} → FY {text}")

2021-22 | Shape: (73, 42)
Column 2 → FY 2020-21
Column 22 → FY 2021-22
2022-23 | Shape: (72, 62)
Column 2 → FY 2020-21
Column 22 → FY 2021-22
Column 42 → FY 2022-23
2023-24 | Shape: (76, 82)
Column 2 → FY 2020-21
Column 22 → FY 2021-22
Column 42 → FY 2022-23
Column 62 → FY 2023-24
2024-25 | Shape: (79, 82)
Column 2 → FY 2021-22
Column 22 → FY 2022-23
Column 42 → FY 2023-24
Column 62 → FY 2024-25


----

## 9. Final Source Selection for Five-Year Life Claims Dataset

The required five-year Life Claims dataset is constructed from the most appropriate available historical blocks across IRDAI Handbook source sheets.

FY2020-21 is sourced from the historical FY2020-21 block in the 2021-22 Handbook, while FY2021-22 to FY2024-25 are sourced from the corresponding available historical/current blocks.

Source provenance will be preserved for every extracted record.

In [25]:
life_source_map = {
    "2020-21": {
        "sheet": "2021_22_Life_Claims",
        "start_col": 2
    },
    "2021-22": {
        "sheet": "2022_23_Life_Claims",
        "start_col": 22
    },
    "2022-23": {
        "sheet": "2023_24_Life_Claims",
        "start_col": 42
    },
    "2023-24": {
        "sheet": "2024_25_Life_Claims",
        "start_col": 42
    },
    "2024-25": {
        "sheet": "2024_25_Life_Claims",
        "start_col": 62
    }
}

life_source_map

{'2020-21': {'sheet': '2021_22_Life_Claims', 'start_col': 2},
 '2021-22': {'sheet': '2022_23_Life_Claims', 'start_col': 22},
 '2022-23': {'sheet': '2023_24_Life_Claims', 'start_col': 42},
 '2023-24': {'sheet': '2024_25_Life_Claims', 'start_col': 42},
 '2024-25': {'sheet': '2024_25_Life_Claims', 'start_col': 62}}

----

## 10. Raw Life Claims Block Extraction

The selected five-year source blocks are extracted without cleaning, standardization, imputation, or calculation.

Original source values are preserved, including blanks, hyphens, percentages, sector totals, and insurer names. Source handbook and source sheet information are retained for provenance.

In [ ]:
for fy, info in life_source_map.items():

    df = pd.read_excel(
        source_path,
        sheet_name=info["sheet"],
        header=None
    )

    start = info["start_col"]
    end = start + 20

    print("=" * 70)
    print(f"FY: {fy}")
    print(f"Source: {info['sheet']}")
    print(f"Columns: {start} to {end - 1}")
    print(f"Shape of selected block: {df.iloc[:, start:end].shape}")

FY: 2020-21
Source: 2021_22_Life_Claims
Columns: 2 to 21
Shape of selected block: (73, 20)
FY: 2021-22
Source: 2022_23_Life_Claims
Columns: 22 to 41
Shape of selected block: (72, 20)
FY: 2022-23
Source: 2023_24_Life_Claims
Columns: 42 to 61
Shape of selected block: (76, 20)
FY: 2023-24
Source: 2024_25_Life_Claims
Columns: 42 to 61
Shape of selected block: (79, 20)
FY: 2024-25
Source: 2024_25_Life_Claims
Columns: 62 to 81
Shape of selected block: (79, 20)


----

## 11. Insurer-Level Raw Record Identification

The source tables contain insurer records, sector headings, sector totals, grand totals, percentage rows, and notes.

For the raw insurer-level dataset, rows containing a numeric S.No. are identified as insurer observations. No values are cleaned, standardized, converted, or recalculated at this stage.

In [27]:
life_raw_insurer = {}

for fy, info in life_source_map.items():

    df = pd.read_excel(
        source_path,
        sheet_name=info["sheet"],
        header=None
    )

    start = info["start_col"]
    end = start + 20

    # Select insurer identifier + required 20-column FY block
    selected = df.iloc[:, [0, 1] + list(range(start, end))].copy()

    # Actual insurer rows have numeric S.No.
    insurer_mask = pd.to_numeric(
        selected.iloc[:, 0],
        errors="coerce"
    ).notna()

    insurer_data = selected.loc[insurer_mask].copy()

    life_raw_insurer[fy] = insurer_data

    print(
        fy,
        "→",
        insurer_data.shape,
        "insurer rows"
    )

2020-21 → (24, 22) insurer rows
2021-22 → (24, 22) insurer rows
2022-23 → (26, 22) insurer rows
2023-24 → (26, 22) insurer rows
2024-25 → (26, 22) insurer rows


In [28]:
for fy, df in life_raw_insurer.items():
    print("\n", "=" * 50)
    print(fy)
    print(df.iloc[:, :2].to_string(index=False))


2020-21
 0                     1
 1                  LIC 
 2 Aditya Birla Sun Life
 3                 Aegon
 4         Ageas Federal
 5                 Aviva
 6         Bajaj Allianz
 7           Bharti Axa 
 8       Canara HSBC OBC
 9       Edelweiss Tokio
10            Exide Life
11       Future Generali
12             HDFC Life
13      ICICI Prudential
14           India First
15       Kotak Mahindra 
16              Max Life
17          PNB Met Life
18        Pramerica Life
19       Reliance Nippon
20                Sahara
21             SBI Life 
22               Shriram
23            Star Union
24              Tata AIA

2021-22
 0                     1
 1                  LIC 
 2 Aditya Birla Sun Life
 3                 Aegon
 4         Ageas Federal
 5                 Aviva
 6         Bajaj Allianz
 7           Bharti Axa 
 8       Canara HSBC OBC
 9       Edelweiss Tokio
10            Exide Life
11       Future Generali
12             HDFC Life
13      ICICI Prudential
14     

In [29]:
for fy, df in life_raw_insurer.items():
    print("=" * 60)
    print(f"FY: {fy}")
    
    print(
        df.iloc[:, [0, 1]].to_string(index=False)
    )

FY: 2020-21
 0                     1
 1                  LIC 
 2 Aditya Birla Sun Life
 3                 Aegon
 4         Ageas Federal
 5                 Aviva
 6         Bajaj Allianz
 7           Bharti Axa 
 8       Canara HSBC OBC
 9       Edelweiss Tokio
10            Exide Life
11       Future Generali
12             HDFC Life
13      ICICI Prudential
14           India First
15       Kotak Mahindra 
16              Max Life
17          PNB Met Life
18        Pramerica Life
19       Reliance Nippon
20                Sahara
21             SBI Life 
22               Shriram
23            Star Union
24              Tata AIA
FY: 2021-22
 0                     1
 1                  LIC 
 2 Aditya Birla Sun Life
 3                 Aegon
 4         Ageas Federal
 5                 Aviva
 6         Bajaj Allianz
 7           Bharti Axa 
 8       Canara HSBC OBC
 9       Edelweiss Tokio
10            Exide Life
11       Future Generali
12             HDFC Life
13      ICICI Prudential
1

In [30]:
for fy, df in life_raw_insurer.items():
    names = df.iloc[:, 1].astype(str).str.strip()
    duplicates = names[names.duplicated(keep=False)]
    
    if len(duplicates) > 0:
        print(f"\n{fy} duplicate insurer names:")
        print(duplicates.tolist())

duplicates.tolist()

[]

In [31]:
life_2021_22.iloc[19:28, :22].T

,19,20,21,22,23,24,25,26,27
0,7,NaN,8,NaN,9,NaN,10,NaN,11
1,Bharti Axa,NaN,Canara HSBC OBC,NaN,Edelweiss Tokio,NaN,Exide Life,NaN,Future Generali
2,2,NaN,2,NaN,0,NaN,38,NaN,3
3,1.153392,NaN,2,NaN,0,NaN,9.048578,NaN,0.31135
4,1891,NaN,1897,NaN,502,NaN,5014,NaN,1223
5,106.444937,NaN,166.537924,NaN,52.089138,NaN,173.655439,NaN,55.473135
6,1893,1,1899,1.0,502,1,5052,1,1226
7,107.598328,1,168.537924,1,52.089138,1,182.704017,1,55.784485
8,1875,0.990491,1844,0.971037,487,0.97012,4978,0.985352,1163
9,106.035205,0.985473,156.075726,0.926057,45.82828,0.879805,170.430141,0.932821,48.110547


In [32]:
life_2021_22.iloc[19:28, 22:42].T

,19,20,21,22,23,24,25,26,27
22,0,NaN,25,NaN,2,NaN,63,NaN,8
23,0,NaN,5.867974,NaN,3,NaN,8.784217,NaN,2.711306
24,3203,NaN,2788,NaN,993,NaN,7216,NaN,1656
25,243.588059,NaN,282.233418,NaN,156.863104,NaN,309.833294,NaN,83.981791
26,3203,1,2813,1,995,1,7279,1,1664
27,243.588059,1,288.101392,1.0,159.863104,1,318.617512,1,86.693097
28,3174,0.990946,2769,0.984358,976,0.980905,7213,0.990933,1600
29,234.719895,0.963594,277.79491,0.964226,152.062588,0.951205,306.808779,0.962938,77.673195
30,27,0.00843,41,0.014575,19,0.019095,7,0.000962,64
31,2.044654,0.008394,7.806482,0.027096,7.800516,0.048795,0.320898,0.001007,9.019901


In [35]:
life_2021_22.iloc[38:56, [0, 1]]

,0,1
38,NaN,NaN
39,17,PNB Met Life
40,NaN,NaN
41,18,Pramerica Life
42,NaN,NaN
43,19,Reliance Nippon
44,NaN,NaN
45,20,Sahara
46,NaN,NaN
47,21,SBI Life


In [36]:
life_2021_22.iloc[38:56, :22].T

,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55
0,NaN,17,NaN,18,NaN,19,NaN,20,NaN,21,NaN,22,NaN,23,NaN,24,NaN,NaN
1,NaN,PNB Met Life,NaN,Pramerica Life,NaN,Reliance Nippon,NaN,Sahara,NaN,SBI Life,NaN,Shriram,NaN,Star Union,NaN,Tata AIA,NaN,Private Sector Total
2,NaN,0,NaN,2,NaN,2,NaN,46,NaN,36,NaN,5,NaN,3,NaN,0,NaN,319
3,NaN,0,NaN,0.221508,NaN,0.571959,NaN,0.579634,NaN,7.776073,NaN,0.119444,NaN,0.721065,NaN,0,NaN,104.95815
4,NaN,5315,NaN,645,NaN,9414,NaN,839,NaN,34183,NaN,3681,NaN,1632,NaN,4648,NaN,154012
5,NaN,353.158002,NaN,29.286068,NaN,215.421583,NaN,8.677096,NaN,1614.551157,NaN,123.545609,NaN,77.404959,NaN,546.33,NaN,9016.990454
6,1.0,5315,1,647,1.0,9416,1,885,1,34219,1,3686,1,1635,1,4648,1,154331
7,1,353.158002,1,29.507576,1.0,215.993543,1,9.256731,1.0,1622.32723,1,123.665053,1,78.126024,1,546.33,1,9121.948604
8,0.993517,5218,0.98175,638,0.98609,9274,0.984919,860,0.971751,31855,0.930916,3506,0.951167,1569,0.959633,4556,0.980207,149734
9,0.954191,331.703223,0.939249,28.625236,0.970098,205.261737,0.950314,9.022812,0.97473,1398.775249,0.862203,95.810658,0.774759,72.494128,0.927913,478.4,0.875661,8125.919442


In [37]:
life_raw_insurer["2020-21"].iloc[:, [0, 1]].to_string(index=False)

' 0                     1\n 1                  LIC \n 2 Aditya Birla Sun Life\n 3                 Aegon\n 4         Ageas Federal\n 5                 Aviva\n 6         Bajaj Allianz\n 7           Bharti Axa \n 8       Canara HSBC OBC\n 9       Edelweiss Tokio\n10            Exide Life\n11       Future Generali\n12             HDFC Life\n13      ICICI Prudential\n14           India First\n15       Kotak Mahindra \n16              Max Life\n17          PNB Met Life\n18        Pramerica Life\n19       Reliance Nippon\n20                Sahara\n21             SBI Life \n22               Shriram\n23            Star Union\n24              Tata AIA'

In [38]:
for fy, df in life_raw_insurer.items():
    print(f"{fy}:")
    print("Rows:", len(df))
    print("Unique insurers:", df.iloc[:, 1].nunique())
    print()

2020-21:
Rows: 24
Unique insurers: 24

2021-22:
Rows: 24
Unique insurers: 24

2022-23:
Rows: 26
Unique insurers: 26

2023-24:
Rows: 26
Unique insurers: 26

2024-25:
Rows: 26
Unique insurers: 26



----

## 12. Cross-Year Metric Structure Validation

The selected five-year Life Claims blocks are validated to ensure that the same claim-status and duration-wise fields are present in the same order across the selected source blocks.

No cleaning, standardization, or calculation is performed in this step.

In [39]:
for fy, info in life_source_map.items():

    df = pd.read_excel(
        source_path,
        sheet_name=info["sheet"],
        header=None
    )

    start = info["start_col"]
    end = start + 20

    print("=" * 80)
    print(f"FY: {fy}")
    print(f"Source: {info['sheet']}")
    print(f"Columns: {start} to {end - 1}")
    print("-" * 80)

    for col in range(start, end):
        print(
            f"{col} | "
            f"{df.iloc[2, col] if pd.notna(df.iloc[2, col]) else ''} | "
            f"{df.iloc[3, col] if pd.notna(df.iloc[3, col]) else ''}"
        )

    print()

FY: 2020-21
Source: 2021_22_Life_Claims
Columns: 2 to 21
--------------------------------------------------------------------------------
2 | Claims pending at start of the period | No. of Policies
3 |  | Benefit Amount (₹crore)
4 | Claims intimated / booked | No. of Policies
5 |  | Benefit Amount (₹crore)
6 | Total Claims | No. of Policies
7 |  | Benefit Amount (₹crore)
8 | Claims Paid | No. of Policies
9 |  | Benefit Amount (₹crore)
10 | Claims Repudiated  | No. of Policies
11 |  | Benefit Amount (₹crore)
12 | Claims Rejected | No. of Policies
13 |  | Benefit Amount (₹crore)
14 | Claims Unclaimed | No. of Policies
15 |  | Benefit Amount (₹crore)
16 | Claims pending at end of the period | No. of Policies
17 |  | Benefit Amount (₹crore)
18 | Break up of claims pending -- duration wise (Policies) | < 3 months
19 |  | 3 - < 6 months
20 |  | 6 - <1 yr
21 |  | > 1 yr

FY: 2021-22
Source: 2022_23_Life_Claims
Columns: 22 to 41
-----------------------------------------------------------------

In [40]:
for fy, info in life_source_map.items():

    df = pd.read_excel(
        source_path,
        sheet_name=info["sheet"],
        header=None
    )

    start = info["start_col"]

    print("=" * 70)
    print(f"{fy} | {info['sheet']} | Start Column: {start}")

    for offset in range(20):
        col = start + offset

        row2 = df.iloc[2, col]
        row3 = df.iloc[3, col]

        row2 = "" if pd.isna(row2) else str(row2).strip()
        row3 = "" if pd.isna(row3) else str(row3).strip()

        print(f"{offset:02d} → {row2} | {row3}")

2020-21 | 2021_22_Life_Claims | Start Column: 2
00 → Claims pending at start of the period | No. of Policies
01 →  | Benefit Amount (₹crore)
02 → Claims intimated / booked | No. of Policies
03 →  | Benefit Amount (₹crore)
04 → Total Claims | No. of Policies
05 →  | Benefit Amount (₹crore)
06 → Claims Paid | No. of Policies
07 →  | Benefit Amount (₹crore)
08 → Claims Repudiated | No. of Policies
09 →  | Benefit Amount (₹crore)
10 → Claims Rejected | No. of Policies
11 →  | Benefit Amount (₹crore)
12 → Claims Unclaimed | No. of Policies
13 →  | Benefit Amount (₹crore)
14 → Claims pending at end of the period | No. of Policies
15 →  | Benefit Amount (₹crore)
16 → Break up of claims pending -- duration wise (Policies) | < 3 months
17 →  | 3 - < 6 months
18 →  | 6 - <1 yr
19 →  | > 1 yr
2021-22 | 2022_23_Life_Claims | Start Column: 22
00 → Claims pending at start of the period | No. of Policies
01 →  | Benefit Amount (₹crore)
02 → Claims intimated / booked | No. of Policies
03 →  | Benefit 

In [41]:
expected_structure = [
    "Pending Start - Count",
    "Pending Start - Amount",
    "Intimated/Booked - Count",
    "Intimated/Booked - Amount",
    "Total Claims - Count",
    "Total Claims - Amount",
    "Claims Paid - Count",
    "Claims Paid - Amount",
    "Claims Repudiated - Count",
    "Claims Repudiated - Amount",
    "Claims Rejected - Count",
    "Claims Rejected - Amount",
    "Claims Unclaimed - Count",
    "Claims Unclaimed - Amount",
    "Pending End - Count",
    "Pending End - Amount",
    "Ageing - <3 months",
    "Ageing - 3-<6 months",
    "Ageing - 6-<1 year",
    "Ageing - >1 year"
]

for fy, info in life_source_map.items():

    df = pd.read_excel(
        source_path,
        sheet_name=info["sheet"],
        header=None
    )

    start = info["start_col"]

    print(f"\n{fy} | {info['sheet']}")

    for i in range(20):

        col = start + i

        row2 = "" if pd.isna(df.iloc[2, col]) else str(df.iloc[2, col]).strip()
        row3 = "" if pd.isna(df.iloc[3, col]) else str(df.iloc[3, col]).strip()

        print(f"{i:02d} | {row2} | {row3}")


2020-21 | 2021_22_Life_Claims
00 | Claims pending at start of the period | No. of Policies
01 |  | Benefit Amount (₹crore)
02 | Claims intimated / booked | No. of Policies
03 |  | Benefit Amount (₹crore)
04 | Total Claims | No. of Policies
05 |  | Benefit Amount (₹crore)
06 | Claims Paid | No. of Policies
07 |  | Benefit Amount (₹crore)
08 | Claims Repudiated | No. of Policies
09 |  | Benefit Amount (₹crore)
10 | Claims Rejected | No. of Policies
11 |  | Benefit Amount (₹crore)
12 | Claims Unclaimed | No. of Policies
13 |  | Benefit Amount (₹crore)
14 | Claims pending at end of the period | No. of Policies
15 |  | Benefit Amount (₹crore)
16 | Break up of claims pending -- duration wise (Policies) | < 3 months
17 |  | 3 - < 6 months
18 |  | 6 - <1 yr
19 |  | > 1 yr

2021-22 | 2022_23_Life_Claims
00 | Claims pending at start of the period | No. of Policies
01 |  | Benefit Amount (₹crore)
02 | Claims intimated / booked | No. of Policies
03 |  | Benefit Amount (₹crore)
04 | Total Claims |

----

## 13. Build Combined Raw Life Claims Dataset

The validated insurer-level Life Claims data for FY2020-21 to FY2024-25 is consolidated into a single raw dataset.

The source values are preserved as extracted from the IRDAI Annual Handbooks. No missing-value treatment, insurer-name standardization, type conversion, or derived metric calculation is performed at this stage.

Financial year and source-sheet information are retained to preserve data lineage and auditability.

In [44]:
life_raw_insurer = {}

for fy, info in life_source_map.items():

    df = pd.read_excel(
        source_path,
        sheet_name=info["sheet"],
        header=None
    )

    start = info["start_col"]

    block = df.iloc[:, start:start + 20].copy()

    insurer_col = df.iloc[:, 1].copy()

    valid_rows = (
        insurer_col.notna()
        & df.iloc[:, 0].apply(
            lambda x: str(x).strip().replace(".", "", 1).isdigit()
        )
    )

    block = block.loc[valid_rows].copy()
    insurer_names = insurer_col.loc[valid_rows].copy()

    # IMPORTANT:
    # Reset source column labels to relative positions
    block.columns = range(20)

    # Add identifiers
    block.insert(0, "Insurer", insurer_names.values)
    block.insert(0, "FY", fy)

    life_raw_insurer[fy] = block.reset_index(drop=True)

    print(f"{fy}: {block.shape}")

2020-21: (24, 22)
2021-22: (24, 22)
2022-23: (26, 22)
2023-24: (26, 22)
2024-25: (26, 22)


In [45]:
life_raw_combined = pd.concat(
    life_raw_insurer.values(),
    ignore_index=True
)

print("Final shape:", life_raw_combined.shape)

Final shape: (126, 22)


In [46]:
life_raw_combined.columns = [
    "FY",
    "Insurer",
    "pending_start_count",
    "pending_start_amount",
    "intimated_count",
    "intimated_amount",
    "total_claims_count",
    "total_claims_amount",
    "paid_count",
    "paid_amount",
    "repudiated_count",
    "repudiated_amount",
    "rejected_count",
    "rejected_amount",
    "unclaimed_count",
    "unclaimed_amount",
    "pending_end_count",
    "pending_end_amount",
    "pending_lt_3m",
    "pending_3_to_6m",
    "pending_6m_to_1y",
    "pending_gt_1y"
]

life_raw_combined.head()

,FY,Insurer,pending_start_count,pending_start_amount,intimated_count,intimated_amount,total_claims_count,total_claims_amount,paid_count,paid_amount,repudiated_count,repudiated_amount,rejected_count,rejected_amount,unclaimed_count,unclaimed_amount,pending_end_count,pending_end_amount,pending_lt_3m,pending_3_to_6m,pending_6m_to_1y,pending_gt_1y
0,2020-21,LIC,5875,349.69,941101,18755.65,946976,19105.34,933889,18295.58,6531,276.93,2934,3.92,1897,236.49,1725,292.42,792,933,0,0
1,2020-21,Aditya Birla Sun Life,19,3.802883,6455,468.846357,6474,472.64924,6347,440.264288,116,28.747045,0,0,0,0,11,3.637908,10,1,0,0
2,2020-21,Aegon,0,0,401,107.44,401,107.44,398,105.98,3,1.46,0,0,0,0,0,-0.0,0,0,0,0
3,2020-21,Ageas Federal,5,1.255,1800,87.051683,1805,88.306683,1716,73.848647,38,7.711034,1,0.048501,0,0,50,6.698502,50,0,0,0
4,2020-21,Aviva,5,0.778359,1050,116.371251,1055,117.14961,1034,111.572118,21,5.577492,0,0,0,0,0,0.0,0,0,0,0


In [47]:
life_raw_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   FY                    126 non-null    str   
 1   Insurer               126 non-null    str   
 2   pending_start_count   122 non-null    object
 3   pending_start_amount  122 non-null    object
 4   intimated_count       122 non-null    object
 5   intimated_amount      122 non-null    object
 6   total_claims_count    122 non-null    object
 7   total_claims_amount   122 non-null    object
 8   paid_count            122 non-null    object
 9   paid_amount           122 non-null    object
 10  repudiated_count      122 non-null    object
 11  repudiated_amount     122 non-null    object
 12  rejected_count        122 non-null    object
 13  rejected_amount       122 non-null    object
 14  unclaimed_count       122 non-null    object
 15  unclaimed_amount      122 non-null    object
 16  p

----

## 14. Raw Dataset Integrity Audit

The consolidated raw Life Claims dataset is subjected to final integrity checks before downstream cleaning.

The audit verifies record counts, financial-year coverage, insurer identifiers, duplicate insurer-year combinations, and the presence of source-level missing markers.

No values are modified during this audit.

In [48]:
# 1. Overall shape
print("Shape:", life_raw_combined.shape)

# 2. Records by financial year
print("\nRecords by FY:")
print(life_raw_combined["FY"].value_counts().sort_index())

# 3. Missing identifiers
print("\nMissing FY:", life_raw_combined["FY"].isna().sum())
print("Missing Insurer:", life_raw_combined["Insurer"].isna().sum())

# 4. Duplicate insurer-year combinations
duplicates = life_raw_combined.duplicated(
    subset=["FY", "Insurer"]
).sum()

print("\nDuplicate FY + Insurer:", duplicates)

# 5. Source '-' markers
source_cols = life_raw_combined.columns[2:]

dash_counts = (
    life_raw_combined[source_cols]
    .astype(str)
    .apply(lambda col: col.str.strip().eq("-").sum())
)

print("\n'-' markers by column:")
print(dash_counts)

# 6. Actual NaN values
print("\nActual NaN values by column:")
print(life_raw_combined.isna().sum())

Shape: (126, 22)

Records by FY:
FY
2020-21    24
2021-22    24
2022-23    26
2023-24    26
2024-25    26
Name: count, dtype: int64

Missing FY: 0
Missing Insurer: 0

Duplicate FY + Insurer: 0

'-' markers by column:
pending_start_count      0
pending_start_amount     0
intimated_count          0
intimated_amount         0
total_claims_count       0
total_claims_amount      0
paid_count               0
paid_amount              0
repudiated_count         0
repudiated_amount        0
rejected_count          20
rejected_amount         20
unclaimed_count         18
unclaimed_amount        18
pending_end_count        4
pending_end_amount       4
pending_lt_3m            6
pending_3_to_6m         14
pending_6m_to_1y        17
pending_gt_1y           21
dtype: int64

Actual NaN values by column:
FY                      0
Insurer                 0
pending_start_count     4
pending_start_amount    4
intimated_count         4
intimated_amount        4
total_claims_count      4
total_claims_amoun

In [49]:
life_raw_combined.isna().sum()

FY                      0
Insurer                 0
pending_start_count     4
pending_start_amount    4
intimated_count         4
intimated_amount        4
total_claims_count      4
total_claims_amount     4
paid_count              4
paid_amount             4
repudiated_count        4
repudiated_amount       4
rejected_count          4
rejected_amount         4
unclaimed_count         4
unclaimed_amount        4
pending_end_count       4
pending_end_amount      4
pending_lt_3m           4
pending_3_to_6m         4
pending_6m_to_1y        4
pending_gt_1y           4
dtype: int64

In [50]:
life_raw_combined.tail()

,FY,Insurer,pending_start_count,pending_start_amount,intimated_count,intimated_amount,total_claims_count,total_claims_amount,paid_count,paid_amount,repudiated_count,repudiated_amount,rejected_count,rejected_amount,unclaimed_count,unclaimed_amount,pending_end_count,pending_end_amount,pending_lt_3m,pending_3_to_6m,pending_6m_to_1y,pending_gt_1y
121,2024-25,Sahara,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
122,2024-25,SBI Life,118,23.580493,44831,2598.751947,44949,2622.332439,44205,2497.939508,524,76.51071,0,0,55,7.610102,165,40.27212,106,23,34,2
123,2024-25,Shriram,6,0.294771,4846,180.195625,4852,180.490395,4770,147.476986,68,22.6648,7,10.150834,0,0,7,0.197776,7,0,0,0
124,2024-25,Star Union,3,0.805,2410,144.384758,2413,145.189758,2385,140.738808,26,3.8752,0,0,0,0,2,0.57575,2,0,0,0
125,2024-25,Tata AIA,2,8.063127,8575,1238.931212,8577,1246.994339,8526,1226.742754,49,17.753274,0,0,0,0,2,2.49831,2,0,0,0


----

## 15. Save Validated Raw Life Claims Dataset

The validated Life Claims dataset covering FY2020-21 to FY2024-25 is saved as a separate downstream raw-extraction file.

The original IRDAI source files remain immutable. This extracted dataset preserves the source values and will serve as the input for the cleaning and standardization stage.

In [51]:
from pathlib import Path

raw_output_path = Path("../data/raw/Life_Claims_Raw_Extracted.csv")

life_raw_combined.to_csv(
    raw_output_path,
    index=False
)

print(f"Saved successfully: {raw_output_path}")
print("Shape:", life_raw_combined.shape)

Saved successfully: ..\data\raw\Life_Claims_Raw_Extracted.csv
Shape: (126, 22)


In [52]:
life_raw_check = pd.read_csv(raw_output_path)

print("Reloaded shape:", life_raw_check.shape)
print("Columns:", life_raw_check.columns.tolist())

Reloaded shape: (126, 22)
Columns: ['FY', 'Insurer', 'pending_start_count', 'pending_start_amount', 'intimated_count', 'intimated_amount', 'total_claims_count', 'total_claims_amount', 'paid_count', 'paid_amount', 'repudiated_count', 'repudiated_amount', 'rejected_count', 'rejected_amount', 'unclaimed_count', 'unclaimed_amount', 'pending_end_count', 'pending_end_amount', 'pending_lt_3m', 'pending_3_to_6m', 'pending_6m_to_1y', 'pending_gt_1y']


----

## Final Data Quality Check

The extracted dataset is validated before being exported for downstream
cleaning.

The checks confirm:

- Expected financial-year coverage
- Expected insurer-year structure
- No duplicate insurer-year records
- Required analytical columns are present
- Dataset dimensions are consistent

In [ ]:
print("Final Shape:", df.shape)
print("Financial Years:", sorted(df['FY'].unique()))
print("Unique Insurers:", df['Insurer'].nunique())
print(
    "Duplicate Insurer-FY Rows:",
    df.duplicated(['Insurer', 'FY']).sum()
)
print("Columns:", df.columns.tolist())

----

## 16. Notebook Summary — Life Claims Source Audit

The Life Claims source-audit and raw extraction stage has been completed successfully.

### Final Dataset
- Financial years covered: FY2020-21 to FY2024-25
- Insurer-year records: 126
- Variables: 21
- Missing FY identifiers: 0
- Missing insurer identifiers: 0
- Duplicate FY–Insurer combinations: 0

The selected insurer-wise Life Claims source blocks were validated across all five financial years, including changes in source table numbering and column positions across different Annual Handbooks.

The extracted data has been consolidated into a separate raw dataset while preserving the original source values and maintaining data lineage.

### Data Integrity Principle

The original IRDAI source files remain immutable. No cleaning, standardization, missing-value treatment, or derived calculations were performed in this notebook.

The validated raw dataset will serve as the input for the subsequent Life Claims Data Cleaning & Standardization stage.

**Output:** `Life_Claims_Raw_Extracted.csv`

**Status:** ✅ Source Audit & Raw Extraction Complete

---

## Conclusion

The source Life Claims data has been audited, the required insurer-level
records have been extracted, and the financial-year structure has been
validated.

The resulting dataset is preserved as a separate raw analytical extract
and will be used as the input for Phase 2 — Data Cleaning & Validation.

No cleaning or feature engineering is performed in this phase.

----

# Thank you ! 